# Team A — Expanded-data model training (new experiment)

This executed copy is separate from the submitted notebook/report. Original
datasets and saved models remain unchanged. Its training file contains 6,325,651 transactions:
original January 2019–June 2020 training data plus newly generated full years 2021–2025.
The 2026 transactions are reserved in a separate fraudTest.csv.
The local folder name expanded_training_2021_2026 describes the generated period, not the earliest original training date.
Dataset CSVs are kept locally and are not included in this GitHub repository.

Year files were available for exploratory scoring before training.
The 2026 holdout is excluded from fitting/validation, but is not claimed as a
pristine unseen experiment. This notebook reports validation results only.
Training is separate from the demo and saves new model files.


## 1. Locate the current project code

Run locally from the Phase III folder, or in Colab clone the current integration
branch. The old executed notebook embeds historical v2 code; this copy imports
the current v3 modules instead. Colab needs the combined CSV copied/uploaded to
its runtime or mounted Drive. Repository code does not include the CSVs.


In [1]:
from pathlib import Path
import os, sys, subprocess, json, hashlib
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
IN_COLAB = Path('/content').exists()
if IN_COLAB:
    checkout = Path('/content/CAI2820C_FraudShield')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--branch', 'phase-iii-history-context',
                        '--single-branch', 'https://github.com/MCDR-Code/CAI2820C_FraudShield.git',
                        str(checkout)], check=True)
    PROJECT = checkout / 'Part_III_Modeling_Demo'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements.txt')], check=True)
else:
    PROJECT = Path.cwd()
    if not (PROJECT / 'train.py').exists():
        PROJECT = PROJECT / 'Part_III_Modeling_Demo'
assert (PROJECT / 'train.py').exists(), 'Set PROJECT to the Phase III folder.'
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
from train import train
from core import load_bundle
from history import HISTORY_VERSION
from shared.project_data import sha256
import pandas as pd
print('Project:', PROJECT)
print('History version:', HISTORY_VERSION)


Project: G:\1MDC_Education\CAI2820C_Artificial_Intelligence_App\CAI2820C_FraudShield\Part_III_Modeling_Demo
History version: strict-timestamp-groups-v3


## 2. Data location and experiment settings

The combined training file is `data/expanded_training_2021_2026/fraudTrain.csv`.
The prepared split contains training through 2025 and a separate 2026 test file. For Colab, mount Drive and
change DATA_DIR below to the folder containing that file. Optional Drive mounting:
`from google.colab import drive; drive.mount('/content/drive')`.

All three algorithms are selectable here. The transaction-model experiment fits
Logistic Regression, Random Forest and Gradient Boosting on identical fitting/
validation rows. The second experiment fits Gradient Boosting with v3 history.
Remove an experiment/model from this dictionary if you do not want to fit it.
Use a new RUN_NAME for each run; existing model folders are never overwritten.


In [2]:
DATA_DIR = PROJECT / 'data/expanded_training_2021_2026'
RUN_NAME = 'expanded_through_2025_run1'
ARTIFACTS = PROJECT / 'artifacts' / RUN_NAME
EXPERIMENTS = {
    'transaction_models': (False, ('Logistic Regression', 'Random Forest', 'Gradient Boosting')),
    'history_v3': (True, ('Gradient Boosting',)),
}
assert (DATA_DIR / 'fraudTrain.csv').exists(), 'Set DATA_DIR or copy/upload fraudTrain.csv first.'
manifest_path = DATA_DIR / 'fraudTrain.join.json'
if not manifest_path.exists():
    manifest_path = DATA_DIR / 'training_data_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    assert sha256(DATA_DIR / 'fraudTrain.csv') == manifest['sha256'], 'Training CSV differs from saved manifest.'
    display(pd.Series({key: manifest.get(key, 'Not recorded in this manifest') for key in ['rows', 'population_cards', 'fraud']}))
print('2026 test present:', (DATA_DIR / 'fraudTest.csv').exists())


rows                                      6325651
population_cards    Not recorded in this manifest
fraud               Not recorded in this manifest
dtype: object

2026 test present: True


## 3. Train once and save new models

The earliest approximately 80% of transactions are fitting data; the latest
approximately 20% are validation. Transactions sharing a timestamp remain on the
same side of the boundary. History uses only strictly earlier transactions,
including earlier fitting-period context during validation. No target labels
are used as features. Thresholds/model selection use validation only.

The explicit expanded-data setting permits this new dataset size. The 2026
file is test only. Original training defaults retain their original size checks.
If a new fraudTest.csv is supplied, its labels remain unread during training and
its chronology/transaction-ID separation are checked. Do not put the old test
file in this folder. The execution outputs below record the completed expanded training run.


In [3]:
experiments = {}
for name, (use_history, models) in EXPERIMENTS.items():
    output = ARTIFACTS / name
    locked, comparison = train(DATA_DIR, output, history=use_history,
                               candidates_to_fit=models, expanded_data=True)
    experiments[name] = {'metadata': locked, 'comparison': comparison}
    display(comparison)


Loading full development data; test labels remain unread.


Fitting Logistic Regression on 5,060,519 training rows...


{
  "model": "Logistic Regression",
  "fit_seconds": 63.986266000021715,
  "threshold": 0.9762790911118733,
  "rows": 1265132,
  "fraud": 2267,
  "accuracy": 0.9949760183127137,
  "precision": 0.16522023906991976,
  "recall": 0.4450816056462285,
  "f1": 0.2409839980893241,
  "average_precision": 0.08132525623515935,
  "roc_auc": 0.8949943844908936,
  "tn": 1257767,
  "fp": 5098,
  "fn": 1258,
  "tp": 1009,
  "review_rate": 0.0048271642800909315,
  "targets_met": false
}


Fitting Random Forest on 5,060,519 training rows...


{
  "model": "Random Forest",
  "fit_seconds": 793.2906730000395,
  "threshold": 0.564626197282559,
  "rows": 1265132,
  "fraud": 2267,
  "accuracy": 0.9936133146580752,
  "precision": 0.1922710428798306,
  "recall": 0.8010586678429643,
  "f1": 0.31010928961748635,
  "average_precision": 0.6392112721711457,
  "roc_auc": 0.9864932767449252,
  "tn": 1255236,
  "fp": 7629,
  "fn": 451,
  "tp": 1816,
  "review_rate": 0.007465624140405902,
  "targets_met": true
}


...

Loading full development data; test labels remain unread.


Fitting Gradient Boosting on 5,060,519 training rows...


{
  "model": "Gradient Boosting",
  "fit_seconds": 172.21340379997855,
  "threshold": 0.9719469407324164,
  "rows": 1265132,
  "fraud": 2267,
  "accuracy": 0.99975338541749,
  "precision": 0.9587048334115439,
  "recall": 0.9011910013233349,
  "f1": 0.9290586630286494,
  "average_precision": 0.9712747177485138,
  "roc_auc": 0.999552126409567,
  "tn": 1262777,
  "fp": 88,
  "fn": 224,
  "tp": 2043,
  "review_rate": 0.0016844092157972449,
  "targets_met": true
}


Locked Gradient Boosting; artifacts: G:\1MDC_Education\CAI2820C_Artificial_Intelligence_App\CAI2820C_FraudShield\Part_III_Modeling_Demo\artifacts\expanded_through_2025_run1\history_v3


,model,fit_seconds,threshold,rows,fraud,accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp,review_rate,targets_met
0,Majority class,0.000000,0.500000,1265132,2267,0.998208,0.000000,0.000000,0.000000,0.001792,0.500000,1262865,0,2267,0,0.000000,False
1,Amount rule,0.000000,505.418200,1265132,2267,0.990584,0.095793,0.504191,0.160997,0.081952,0.863345,1252076,10789,1124,1143,0.009431,False
2,Gradient Boosting,172.213404,0.971947,1265132,2267,0.999753,0.958705,0.901191,0.929059,0.971275,0.999552,1262777,88,224,2043,0.001684,True


## 4. Review validation results and saved-model checks

Precision describes correct alerts; recall describes detected fraud. Inspect
false alerts and missed fraud alongside both measures. Compare these new
experiments on the SAME validation rows. The old Random Forest's published
metrics used different data and cannot directly measure this experiment's gain.
Each experiment saves its selected model, threshold, data fingerprints,
validation comparison and model-sharing manifest. It does not save every fitted
candidate as a separate demo selector entry.


In [4]:
from model_sharing import check_model
for name in experiments:
    print(name)
    display(pd.read_csv(ARTIFACTS / name / 'validation_comparison.csv'))
    print(check_model(ARTIFACTS / name))


transaction_models


,model,fit_seconds,threshold,rows,fraud,accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp,review_rate,targets_met
0,Majority class,0.000000,0.500000,1265132,2267,0.998208,0.000000,0.000000,0.000000,0.001792,0.500000,1262865,0,2267,0,0.000000,False
1,Amount rule,0.000000,505.418200,1265132,2267,0.990584,0.095793,0.504191,0.160997,0.081952,0.863345,1252076,10789,1124,1143,0.009431,False
2,Logistic Regression,63.986266,0.976279,1265132,2267,0.994976,0.165220,0.445082,0.240984,0.081325,0.894994,1257767,5098,1258,1009,0.004827,False
3,Random Forest,793.290673,0.564626,1265132,2267,0.993613,0.192271,0.801059,0.310109,0.639211,0.986493,1255236,7629,451,1816,0.007466,True
4,Gradient Boosting,140.497115,0.929335,1265132,2267,0.994870,0.231053,0.800176,0.358569,0.598999,0.985409,1256828,6037,453,1814,0.006206,True


{'model': 'Gradient Boosting', 'history_version': None, 'features': 78, 'verified_demo_rows': 1000, 'verified_demo_flags': 8, 'model_sha256': '4b31d922944c2ac99bfcae639bc4a0237551926f0d9c21747389cc97fb2842f5', 'status': 'Copy, source, environment and prediction checks passed'}
history_v3


,model,fit_seconds,threshold,rows,fraud,accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp,review_rate,targets_met
0,Majority class,0.000000,0.500000,1265132,2267,0.998208,0.000000,0.000000,0.000000,0.001792,0.500000,1262865,0,2267,0,0.000000,False
1,Amount rule,0.000000,505.418200,1265132,2267,0.990584,0.095793,0.504191,0.160997,0.081952,0.863345,1252076,10789,1124,1143,0.009431,False
2,Gradient Boosting,172.213404,0.971947,1265132,2267,0.999753,0.958705,0.901191,0.929059,0.971275,0.999552,1262777,88,224,2043,0.001684,True


{'model': 'Gradient Boosting', 'history_version': 'strict-timestamp-groups-v3', 'features': 95, 'verified_demo_rows': 993, 'verified_demo_flags': 90, 'model_sha256': '7c9b4dda61c677933e2730f922959838705cc5180cf5240a1acd773b5ef8911e', 'status': 'Copy, source, environment and prediction checks passed'}


## 5. History error analysis and inference parity

These checks use the training file's validation period, not the reserved 2026 test set.
They run only if the history experiment above was selected.


In [5]:
if 'history_v3' in experiments:
    from verify_context import verify
    from analyze_errors import main as analyze_errors
    display(pd.Series(verify(DATA_DIR, ARTIFACTS / 'history_v3')))
    analyze_errors(ARTIFACTS / 'history_v3', DATA_DIR)


history_version                                                        strict-timestamp-groups-v3
context_rows                                                                              5060519
review_rows                                                                               1265132
model_sha256                                    7c9b4dda61c677933e2730f922959838705cc5180cf524...
training_sha256                                 fee5140205817b8e0980c9cd8b1bd016a9b331cee4ccbb...
validation_inference_max_absolute_difference                                                  0.0
official_test_evaluated                                                                     False
threshold                                                                                0.971947
rows                                                                                      1265132
fraud                                                                                        2267
accuracy            

Validation rows 1,265,132; fraud 2267; flagged 2131; missed 224; false alerts 88

By category (sorted by missed fraud):


                  rows  fraud  caught  false_alerts  missed  recall
category                                                           
grocery_pos        566    566     509             0      57   0.899
misc_net           295    295     256             0      39   0.868
shopping_net       576    576     547             0      29   0.950
gas_transport      169    169     152             0      17   0.899
grocery_net         44     44      30             0      14   0.682
misc_pos            59     59      45             0      14   0.763
shopping_pos       239    239     227             0      12   0.950
personal_care       65     65      57             0       8   0.877
entertainment   174472     78      70             3       8   0.897
food_dining     380369     41      33            26       8   0.805
health_fitness  345541     24      18            30       6   0.750
kids_pets        15730     43      38             1       5   0.884
travel              33     33      28           

                       rows  fraud  caught  false_alerts  missed  recall
amt                                                                     
(0.0, 10.0]          383987    165     119            10      46   0.721
(10.0, 50.0]         352698    257     187            27      70   0.728
(50.0, 100.0]        308235     11       8             0       3   0.727
(100.0, 250.0]       184473     59      40            20      19   0.678
(250.0, 500.0]        23421    630     588            15      42   0.933
(500.0, 1000.0]        9115    810     775            11      35   0.957
(1000.0, 1000000.0]    3203    335     326             5       9   0.973

By hour of day:


        rows  fraud  caught  false_alerts  missed  recall
hour                                                     
0       1903    200     170             0      30   0.850
1       1850    223     205             0      18   0.919
2       1958    191     180             0      11   0.942
3       1817    196     187             0       9   0.954
4       1647     14       7             0       7   0.500
5       1705     15       9             0       6   0.600
6       1719     26      20             0       6   0.769
7       1730     13      11             0       2   0.846
8       1786     16      12             0       4   0.750
9       1712     11       9             0       2   0.818
10      1693     12       8             0       4   0.667
11      1776     18      14             0       4   0.778
12    103161     24      14             4      10   0.583
13    103190     20      15             4       5   0.750
14    103810     24      17             2       7   0.708
15    103957  

                       rows  fraud  caught  false_alerts  missed  recall
hist_prior_count                                                        
(0.0, 10.0]               8      8       8             0       0   1.000
(10.0, 100.0]            55     55      55             0       0   1.000
(100.0, 500.0]           85     85      83             0       2   0.976
(500.0, 1000000.0]  1264984   2119    1897            88     222   0.895

By new merchant for the card (1 = never seen before):


                      rows  fraud  caught  false_alerts  missed  recall
hist_new_merchant                                                      
0.0                1220765   2044    1832            85     212   0.896
1.0                  44367    223     211             3      12   0.946

Score distribution of missed fraud vs caught fraud:
        count   mean    std    min    25%    50%    75%    max
flag                                                          
False   224.0  0.767  0.272  0.001  0.656  0.914  0.951  0.971
True   2043.0  0.997  0.005  0.972  0.997  0.999  1.000  1.000

Missed fraud: median amount $23.70 vs caught $711.48
Fraud spread over 218 cards; missed fraud on 116 cards


## 6. Reserved 2026 test and demo integration

fraudTest.csv contains the reserved 2026 transactions. It must never be joined
into fraudTrain.csv or used for threshold/model selection. Freeze choices using
validation before evaluating test labels. Earlier training rows can supply history.
Do not substitute the original old fraudTest.csv from the prior dataset.

Models saved by this notebook are separate new experiment artifacts. They are
not automatically installed over the demo baseline or history model. Preserve
the original Random Forest for comparison and explicitly choose the new model
after examining the run. Transfer trusted saved models with matching code and
package versions; follow Training_and_Model_Sharing.md.
